# RetailX Customer Segmentation & Predictive Analytics

Portfolio version of my Big Data & Analytics project. The workflow uses **K-Means clustering** to discover customer segments, profiles the segments, then trains a **Decision Tree classifier** to assign new customers to those segments.

**Tools:** Python, pandas, scikit-learn, matplotlib, seaborn, Google Colab

## 1. Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

## 2. Load and prepare the data

The original project uses `RetailX.xlsx`. It is intentionally not included in this public portfolio repository.

In [ ]:
retail_df = pd.read_excel("RetailX.xlsx")

model_features = [
    "age", "income", "avg_order_size", "avg_order_freq", "crossbuy",
    "multichannel", "per_sale", "tenure", "return_rate", "married",
    "own_home", "household_size", "loyalty_card", "avg_mktg_cnt"
]

X_cluster = retail_df[model_features]
scaler = StandardScaler()
scaled_features = scaler.fit_transform(X_cluster)

retail_df.head()

## 3. Elbow Method and K-Means

The Elbow Method was used to compare different values of *k*. The project selected **3 clusters** as a practical, interpretable segmentation.

In [ ]:
inertia = []
for k in range(1, 11):
    model_k = KMeans(n_clusters=k, random_state=42, n_init=10)
    model_k.fit(scaled_features)
    inertia.append(model_k.inertia_)

plt.plot(range(1, 11), inertia, marker="o")
plt.xlabel("Number of clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

kmeans = KMeans(n_clusters=3, random_state=0, n_init=10)
retail_df["cluster"] = kmeans.fit_predict(scaled_features)

## 4. Segment profiles

- **Cluster 0 — Frequently Engaged Customers:** frequent shoppers with strong cross-category and multichannel engagement.
- **Cluster 1 — Loyal Premium Customers:** highest average income and longest tenure, with relatively low marketing exposure.
- **Cluster 2 — Big-Ticket Newcomers:** highest average order size, but low purchase frequency and short tenure.

In [ ]:
segment_profile = retail_df.groupby("cluster")[model_features].mean().round(2)
segment_profile

## 5. Decision Tree segment prediction

In [ ]:
X = retail_df[model_features]
y = retail_df["cluster"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train, y_train)
y_pred = tree.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

The submitted project notebook produced **87.25% test accuracy** for this Decision Tree setup.

## 6. Business recommendations

- **Cluster 0:** cross-category bundles, loyalty rewards and consistent multichannel promotions.
- **Cluster 1:** retention, VIP benefits and selective personalized communication.
- **Cluster 2:** onboarding and re-engagement aimed at turning high-value but infrequent shoppers into repeat customers.

## Limitations and next steps

Useful next steps include quarterly re-clustering, comparing the Decision Tree with Random Forest/XGBoost, validating segments through campaign A/B tests, and adding richer behavioural features such as RFM or customer lifetime value.